Model Evaluation: 16x16 Grid of Reference vs Reconstructed Circuits (Qiskit mpl)

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if (root / 'qcgpt').exists():
    sys.path.insert(0, str(root))
elif (root.parent / 'qcgpt').exists():
    sys.path.insert(0, str(root.parent))


In [ ]:
import os, csv
import numpy as np
import torch
import matplotlib.pyplot as plt
from qcgpt.gates import VOCAB, PAD_ID, BOS_CIRC_ID, EOS_CIRC_ID
from qcgpt.models.policy import CircuitPolicy
from qcgpt.data.qiskit_utils import sample_task
from qcgpt.data.specs import build_spec_sequence_batch
from qcgpt.encoding import tokens_to_circuit
from qcgpt.evaluation.metrics import quantum_fidelity_from_spec
from qcgpt.simulators.qiskit_sim import circuit_to_qiskit


In [ ]:
def load_model(ckpt, device):
    model = CircuitPolicy(vocab_size=len(VOCAB)).to(device)
    if ckpt and Path(ckpt).exists():
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state['model_state_dict'])
    model.eval()
    return model

@torch.no_grad()
def reconstruct_seq(model, device, spec_tensor, max_len=32):
    spec_batch_np, spec_pad_mask_np = build_spec_sequence_batch([spec_tensor])
    spec_batch = torch.tensor(spec_batch_np, dtype=torch.float32, device=device)
    spec_pad_mask = torch.tensor(spec_pad_mask_np, dtype=torch.bool, device=device)
    seqs, _ = model.sample_circuit_tokens(spec_batch, spec_pad_mask, BOS_CIRC_ID, EOS_CIRC_ID, max_len=max_len)
    return [t for t in seqs[0].tolist() if t != PAD_ID]

def render_qc_image(qc):
    fig = qc.draw(output='mpl')
    fig.canvas.draw()
    w,h = fig.canvas.get_width_height()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    img = buf.reshape(h,w,4)
    plt.close(fig)
    return img


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = 'model_checkpoints/20251119_015504/transformer_v1_best.pt'
model = load_model(ckpt, device)
out_dir = Path('model_evaluations') / 'notebook_run'
out_dir.mkdir(parents=True, exist_ok=True)
nrows, ncols = 16, 16
csv_path = out_dir / 'fidelities.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f); writer.writerow(['row','col','fid_ref','fid_recon'])
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols*2.0, nrows*2.0))
fig.tight_layout(pad=0.6)
for r in range(nrows):
    for c in range(ncols):
        spec_tensor, ref_circ = sample_task(max_gates=6)
        seq = reconstruct_seq(model, device, spec_tensor, max_len=32)
        cand_circ = tokens_to_circuit(seq)
        fid_ref = quantum_fidelity_from_spec(spec_tensor, ref_circ)
        fid_cand = quantum_fidelity_from_spec(spec_tensor, cand_circ)
        with open(csv_path, 'a', newline='') as f:
            writer = csv.writer(f); writer.writerow([r, c, f'{fid_ref:.6f}', f'{fid_cand:.6f}'])
        img_ref = render_qc_image(circuit_to_qiskit(ref_circ))
        img_cand = render_qc_image(circuit_to_qiskit(cand_circ))
        hmin = min(img_ref.shape[0], img_cand.shape[0])
        pair = np.concatenate([img_ref[:hmin], img_cand[:hmin]], axis=1)
        ax = axes[r][c]; ax.axis('off'); ax.imshow(pair); ax.set_title(f'R {fid_ref:.2f} | C {fid_cand:.2f}', fontsize=6)
fig_path = out_dir / 'circuits_grid.png'
fig.savefig(fig_path, dpi=200)
fig_path, csv_path
